In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from read_hatch import read_hatch

#day = datetime.today().strftime('%Y-%m-%d')
day = '2024-11-19'
sample =  'all' # 'randomsample500' 

# Read data

In [4]:
metadata = pd.read_csv("./data/combined_tech_characteristics.csv", index_col=0)
ts_stats = pd.read_csv(f"./results/timeseries_stats_{sample}_{day}.csv", index_col=0)
params = pd.read_csv(f"./results/fitting_parameters_{sample}_{day}.csv", index_col=0)
hatch = read_hatch('data/HATCH_v1.5_clean.csv')

# Merge data

## Convert categorical variables in metadata to numerical

### Final Material Use: Low, Medium, High

In [3]:
def material_use(fmu):
    if fmu == 'Low Material Use': return 0
    elif fmu == 'Medium Material Use': return 1
    elif fmu == 'High Material Use': return 2
    else: return np.nan

metadata['FMU'] = metadata['Final Material Use'].apply(material_use)

### Need for Customization: Standardized, Mass-Customized, Customized

In [4]:
def customization(nfc):
    if nfc == 'Standardized [1]': return 0
    elif nfc == 'Mass-Customized [2]': return 1
    elif nfc == 'Customized [3]': return 2
    else: return np.nan

metadata['NfC'] = metadata['Need for Customization'].apply(customization)

### Complexity: Simple, Design-Intensive, Complex

In [5]:
def complexity(comp):
    if comp == 'Simple [1]': return 0
    elif comp == 'Design-Intensive [2]': return 1
    elif comp == 'Complex [3]': return 2
    else: return np.nan

metadata['Comp'] = metadata['Complexity'].apply(complexity)

### Type of Adopter: Firms, Individuals, Both

In [6]:
# Break into two dummy variables
metadata['Firm'] = (metadata['Type of Adopter'] == 'Firms [1]') | (metadata['Type of Adopter'] == 'Both [3]')
metadata['Individual'] = (metadata['Type of Adopter'] == 'Individuals [2]') | (metadata['Type of Adopter'] == 'Both [3]')

### Granularity: Low, Medium, High

In [7]:
def granularity(gran):
    if gran == 'Low': return 0
    elif gran == 'Medium': return 1
    elif gran == 'High': return 2
    else: return np.nan

metadata['Gran'] = metadata['Granularity'].apply(granularity)

### Technology Lifetime: Months, Years, Decades

In [8]:
def techlife(tl):
    if tl == 'Months': return 10^-1
    elif tl == 'Years': return 10^0
    elif tl == 'Decades': return 10^1
    else: return np.nan

metadata['TL'] = metadata['Technology Lifetime'].apply(techlife)

### Strict replacement: No, Yes

In [9]:
#one missing value (should fill in or replace with nan)
metadata['SR'] = metadata['Strict replacement'] == 'Yes'

### Broad replacement: No, Yes

In [10]:
#no missing values
metadata['BR'] = metadata['Broad replacement'] == 'Yes'

### Feedstock: No, Yes

In [11]:
def feedstock(fs):
    if fs == 'Yes': return 1
    elif fs == 'No': return 0
    else: return np.nan

metadata['FS'] = metadata['Feedstock'].apply(feedstock)

## Clean data

In [12]:
metadata.replace('n/d', np.nan, inplace=True)
metadata['Granularity Numerical'] = pd.to_numeric(metadata['Granularity Numerical'], errors='coerce')
metadata['Year of Invention']  = pd.to_numeric(metadata['Year of Invention'], errors='coerce')
metadata['Year of First Embodiment of Tech'] = pd.to_numeric(metadata['Year of First Embodiment of Tech'], errors='coerce')
metadata['FirstCommercialYr'] = pd.to_numeric(metadata['FirstCommercialYr'], errors='coerce')

params = params.loc[params['fit_success']]

In [ ]:
count_cats = metadata.merge(hatch, on = 'Technology Name')
count_cats['Category Type'].value_counts()

## Merge metadata and timeseries stats

In [13]:
preds = pd.merge(metadata, ts_stats, how = 'right', on = 'Technology Name')
full_table = pd.merge(preds, params, how = 'right', left_on = 'tech_name', right_on = 'technology')

## Keep only numerical variables

In [14]:
# argument in df.corr below looks only at numerical
# full_table = full_table.drop(['functional_form', 'fit_procedure', 'fit_success', 'Technology Name', 'technology'], axis = 1)

# Correlation table

In [15]:
pred_corr = preds.corr(numeric_only=True)

for i in full_table["functional_form"].unique():
    full_corr = full_table[full_table["functional_form"]==i].corr(numeric_only=True)
    full_corr.to_csv(f"results/correlations/{i}.csv")

In [16]:
# full_table.to_csv(f'full_table_of_predictors_{sample}.csv')

In [22]:
full_table['ar2_rank'] = full_table.groupby('technology')['adj_r_squared'].rank(method='min', ascending=False)
full_table['ar20'] = full_table['adj_r_squared']
full_table.loc[full_table['ar20'] < 0, 'ar20'] = np.nan
full_table['bestlog'] = (full_table['ar2_rank'] == 1) & (full_table['functional_form'] == 'logistic')
full_table['bestlinear'] = (full_table['ar2_rank'] == 1) & (full_table['functional_form'] == 'linear')
full_table['bestexp'] = (full_table['ar2_rank'] == 1) & (full_table['functional_form'] == 'exponential')

best_subset = full_table.loc[full_table['ar2_rank'] == 1]

In [18]:
best_corr = best_subset.corr(numeric_only=True)
best_corr.columns

Index(['Material Use Numerical', 'Year of Invention',
       'Year of First Embodiment of Tech', 'FirstCommercialYr',
       'Granularity Numerical', 'Average lifetime', 'FMU', 'NfC', 'Comp',
       'Gran', 'TL', 'FS', 'n', 'dt_mean', 'dt_std', 'y_median', 'y_mean',
       'y_std', 'y_min', 'y_max', 'y_min_rel_pos', 'y_max_rel_pos',
       'y_pct_mean_drop_after_max', 'y_trend_slope', 'y_trend_pval', 'dy_mean',
       'dy_std', 'dy_cv', 'dy_trend_slope', 'dy_trend_pval', 'ddy_mean',
       'ddy_std', 'ddy_trend_slope', 'ddy_trend_pval', 'ar1', 'ar2',
       'roughness', 'lastm', 'lastm_convexity_sign', 'lastm_convexity_mean',
       'lastm_convexity_pvalue', 'lastm_slope_sign', 'lastm_slope_mean',
       'lastm_lin_pvalue', 'ratio_last1m_v_first', 'fit_success', 'r_squared',
       'adj_r_squared', 'MCp', 'BIC', 'MAPE', 'a', 'b', 'c', 'a_std', 'b_std',
       'c_std', 'd', 'd_std', 'ar2_rank', 'bestlog', 'bestlinear', 'bestexp'],
      dtype='object')

# Regressions

In [23]:
full_table['ar2_diff'] = np.nan
for t in full_table['technology']:
    full_table.loc[full_table['technology'] == t, 'ar2_diff'] = full_table.loc[full_table['technology'] == t, 'adj_r_squared'] - np.min(full_table.loc[full_table['technology'] == t, 'adj_r_squared'])

KeyboardInterrupt: 

In [33]:
regdf = full_table.loc[full_table['functional_form'] == 'logistic',['ar2_rank', 'Gran', 'TL', 'n', 'FMU', 'y_max_rel_pos', 'y_pct_mean_drop_after_max', 'ar1', 'roughness', 'lastm_convexity_mean','lastm_slope_mean']].dropna()

In [34]:
model = smf.ols(formula = 'ar2_rank ~ Gran + TL + FMU + y_max_rel_pos + ar1 + roughness + lastm_convexity_mean', data = regdf, missing = 'drop') #
res = model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:               ar2_rank   R-squared:                       0.116
Model:                            OLS   Adj. R-squared:                  0.115
Method:                 Least Squares   F-statistic:                     110.0
Date:                Tue, 03 Dec 2024   Prob (F-statistic):          4.54e-152
Time:                        14:55:46   Log-Likelihood:                -19426.
No. Observations:                5878   AIC:                         3.887e+04
Df Residuals:                    5870   BIC:                         3.892e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -0.9232 